# Manual-calibrated ER/MR/LR detection

This notebook treats `annotations/manual_er_mr_lr-annot.fif` as immutable calibration data. It measures the lowercase manual labels, calibrates a robust detector, opens a separate editable review copy, and saves lowercase `*_auto` results.

In [10]:
%matplotlib qt
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

notebook_dir = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from manual_calibrated_er_mr_lr import (
    AUTO_LABELS, atomic_save_annotations, build_calibration,
    candidates_to_annotations, detect_components, evaluate_candidates,
    extract_annotation_metrics, load_mat_raw, normalized_review_annotations,
    plot_best_epochs, plot_metric_distributions, review_annotations,
    save_calibration,
)

mne.viz.set_browser_backend("qt")
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

mat_path = project_root / "data" / "raw" / "pig_4_9_14_day_after_SCI.mat"
manual_annotation_path = project_root / "notebooks" / "annotations" / "manual_er_mr_lr-annot.fif"
metrics_dir = project_root / "outputs" / "metrics"
figures_dir = project_root / "outputs" / "figures"
annotations_dir = project_root / "outputs" / "annotations"
for directory in [metrics_dir, figures_dir, annotations_dir]:
    directory.mkdir(parents=True, exist_ok=True)

channel = "GM Right"
baseline_window_ms = (-25.0, -5.0)
browser_scaling = 8.0779357 / 1.5  # 1.5x larger traces than the current MAT browser
skip_interactive_review = os.environ.get("EMG_SKIP_INTERACTIVE_QC", "0") == "1"

In [11]:
# Load the recording and immutable manual calibration annotations.
if not mat_path.exists():
    raise FileNotFoundError(mat_path)
if not manual_annotation_path.exists():
    raise FileNotFoundError(manual_annotation_path)

raw = load_mat_raw(mat_path)
manual_annotations = mne.read_annotations(manual_annotation_path)
manual_counts = pd.Series(manual_annotations.description, dtype=str).value_counts()
required = {"Stimulus_Auto", "er", "mr", "lr"}
missing = required - set(manual_counts.index)
if missing:
    raise ValueError(f"Missing required annotations: {sorted(missing)}")

print("Recording duration, s:", raw.times[-1])
print("Sampling frequency, Hz:", raw.info["sfreq"])
display(manual_counts.rename("count").to_frame())

Recording duration, s: 538.39975
Sampling frequency, Hz: 4000.0


,count
Stimulus_Auto,562
MR_auto,53
ER_auto,50
lr,35
mr,27
er,26
LR_auto,21
BAD boundary,3
EDGE boundary,3


In [12]:
# Measure manual ER/MR/LR waves; point annotations receive inferred bounds.
manual_metrics, manual_polarities, stimulus_times_s = extract_annotation_metrics(
    raw, manual_annotations, channel=channel, baseline_window_ms=baseline_window_ms
)
manual_metrics.to_csv(metrics_dir / "manual_er_mr_lr_metrics.csv", index=False)

quality_summary = (
    manual_metrics.groupby("label")
    .agg(
        total=("label", "size"),
        inferred_bounds=("bounds_inferred", "sum"),
        included=("included_in_calibration", "sum"),
        median_peak_latency_ms=("peak_latency_ms", "median"),
        median_duration_ms=("duration_ms", "median"),
        median_p2p=("p2p_amplitude", "median"),
    )
)
print("Dominant polarities:", manual_polarities)
display(quality_summary)
display(manual_metrics.loc[~manual_metrics.included_in_calibration, ["label", "stimulus_index", "exclusion_reason"]])

Dominant polarities: {'er': 1, 'mr': -1, 'lr': 1}


,total,inferred_bounds,included,median_peak_latency_ms,median_duration_ms,median_p2p
label,,,,,,
er,26,12,24,14.255127,1.5,0.091563
lr,35,2,35,29.746826,3.0,0.082500
mr,27,1,23,18.502686,3.0,0.425313


,label,stimulus_index,exclusion_reason
0,mr,290,peak_latency_ms_outlier
15,mr,308,peak_latency_ms_outlier
18,er,308,peak_latency_ms_outlier;duration_ms_outlier
31,mr,332,peak_latency_ms_outlier
36,er,333,peak_latency_ms_outlier;duration_ms_outlier
50,mr,339,duration_ms_outlier


In [13]:
# Median + scaled-MAD calibration and distributions.
calibration, calibration_summary = build_calibration(manual_metrics, manual_polarities)
calibration_summary.to_csv(metrics_dir / "manual_er_mr_lr_calibration_summary.csv", index=False)
save_calibration(calibration, metrics_dir / "manual_er_mr_lr_calibration.json")

plot_metric_distributions(
    manual_metrics, calibration, figures_dir / "manual_er_mr_lr_metric_distributions.png"
)
plt.show()
display(calibration_summary)

,label,metric,n,median,scaled_mad,lower,upper
0,er,start_latency_ms,24,12.751465,1.115570,9.404756,16.098174
1,er,peak_latency_ms,24,14.130127,1.116294,10.781246,17.479008
2,er,end_latency_ms,24,14.131836,1.114665,10.787842,17.475830
3,er,duration_ms,24,1.500000,0.370650,0.388050,2.611950
4,er,peak_width_ms,24,0.634722,0.321056,0.000000,1.597890
5,er,absolute_peak_amplitude,24,0.315859,0.246714,0.000000,1.056001
6,er,p2p_amplitude,24,0.086094,0.055598,0.000000,0.252886
7,er,prominence,24,0.037344,0.043320,0.000000,0.167303
8,er,auc_abs,24,0.352520,0.277437,0.000000,1.184831
9,er,rms,24,0.301255,0.245701,0.000000,1.038357


In [14]:
# Run the permissive sequential ER -> MR -> 1-3 LR detector.
auto_candidates = detect_components(
    raw, stimulus_times_s, calibration, channel=channel, baseline_window_ms=baseline_window_ms
)
if auto_candidates.empty:
    raise RuntimeError("The calibrated detector found no complete ER/MR/LR sequences")
auto_candidates.to_csv(metrics_dir / "manual_calibrated_candidates_before_review.csv", index=False)

in_sample_performance = evaluate_candidates(manual_metrics, auto_candidates, calibration)
in_sample_performance.to_csv(metrics_dir / "manual_calibrated_in_sample_performance.csv", index=False)
print("Calibration-set performance (not independent validation):")
display(in_sample_performance)
display(pd.Series(auto_candidates.auto_label, dtype=str).value_counts().rename("count").to_frame())

Calibration-set performance (not independent validation):


,label,manual,automatic,matched,missed,extra,precision,recall,match_tolerance_ms
0,er,24,22,20,4,2,0.909091,0.833333,3.348881
1,mr,23,22,20,3,2,0.909091,0.869565,2.260820
2,lr,35,62,25,10,37,0.403226,0.714286,16.642330


,count
auto_label,
lr_auto,440
er_auto,161
mr_auto,161


In [15]:
# Full interactive review: delete, add, move, or resize lowercase *_auto spans.
# When adding an annotation, use exactly er_auto, mr_auto, or lr_auto.
if skip_interactive_review:
    print("Interactive review skipped; all candidates are retained.")
    descriptions = np.asarray(manual_annotations.description, dtype=str)
    keep = (descriptions == "Stimulus_Auto") | np.char.startswith(descriptions, "BAD") | np.char.startswith(descriptions, "EDGE")
    reviewed_annotations = manual_annotations[keep] + candidates_to_annotations(auto_candidates)
else:
    print("Review candidates, then close the browser to continue.")
    reviewed_annotations = review_annotations(
        raw, manual_annotations, auto_candidates, browser_scaling=browser_scaling, block=True
    )

Review candidates, then close the browser to continue.
Using pyopengl with version 3.1.10
Channels marked as bad:
[np.str_('Art 2')]


In [16]:
# Normalize added point marks, save reviewed metrics, and atomically save annotations.
final_annotations, reviewed_metrics = normalized_review_annotations(
    raw, reviewed_annotations, channel=channel
)
reviewed_metrics.to_csv(metrics_dir / "manual_calibrated_candidates_after_review.csv", index=False)

final_annotation_path = annotations_dir / "stimulus_er_mr_lr_manual_calibrated_auto-annot.fif"
saved_counts = atomic_save_annotations(final_annotations, final_annotation_path)
print("Saved:", final_annotation_path)
print("Round-trip validated counts:", saved_counts)

Saved: /Users/sofiaefimochkina/Documents/lab/emg-lr-segmentation/outputs/annotations/stimulus_er_mr_lr_manual_calibrated_auto-annot.fif
Round-trip validated counts: {'Stimulus_Auto': 562, 'lr_auto': 440, 'er_auto': 161, 'mr_auto': 161, 'BAD boundary': 3, 'EDGE boundary': 3}


In [20]:
# Separate best-10 reviewed epochs, ranked by LR prominence then amplitude.
import importlib
import manual_calibrated_er_mr_lr as manual_helpers

manual_helpers = importlib.reload(manual_helpers)
plot_best_epochs = manual_helpers.plot_best_epochs

best_figure, best_10 = plot_best_epochs(
    raw, reviewed_metrics, stimulus_times_s=stimulus_times_s,
    output_path=figures_dir / "manual_calibrated_best_10_epochs.png",
    channel=channel, n_plots=10, time_scale=0.50,
    row_height=2.4, right_edge_padding_ms=0.5,
)
best_10.to_csv(metrics_dir / "manual_calibrated_best_10_epochs.csv", index=False)
display(best_10)
if best_figure is not None:
    plt.show()
else:
    print("No reviewed LR annotations are available for the best-epochs plot.")

,stimulus_index,lr_prominence,lr_amplitude
0,379,0.548125,1.065469
1,477,0.450312,0.524844
2,171,0.438750,0.470313
3,164,0.428750,0.929688
4,371,0.422500,1.137344
5,547,0.416250,0.459844
6,364,0.415312,0.620938
7,515,0.408125,1.271094
8,202,0.406250,1.154375
9,385,0.388437,0.242031
